# 17. CUDA Stream and Asynchrony | CUDA Stream 与异步执行

**难度：** Medium | **环境：** GPU optional | **标签：** `算子编程`, `CUDA Stream`, `异步执行` | **目标人群：** 编译与系统入门者

---

## 本节导读

吞吐受限不一定来自单个计算内核过慢，也可能来自拷贝、计算和同步被排成了不必要的串行链。本节从任务依赖出发，解释 Stream 顺序、事件同步和异步提交怎样共同决定可重叠区间。

完成后，你应能区分主机提交顺序、设备执行顺序和真实依赖，并使用时间线判断重叠是否发生、同步是否暴露在关键路径上。

**关键词：** CUDA Stream、异步执行（asynchrony）、重叠（overlap）

![CUDA Stream、依赖与重叠时间线](../docs/public/01_Hardware_Math_and_Systems/17_stream_dependency_overlap.svg)

---

## 前置阅读

**导语：** 先把 block 和 shared memory 的关系对齐，再看 stream 和异步重叠会更顺。

- [Group 1C: Communication, Parallelism and Fault Tolerance | 1C: 通信、并行与容错](./1C.md)
- [15. CUDA Execution Model | CUDA 执行模型](./15_CUDA_Execution_Model.ipynb)
- [16. Warp Block SharedMemory Basics | Warp、Block 与 Shared Memory 基础](./16_Warp_Block_SharedMemory_Basics.ipynb)


## Q1：CUDA Stream 到底是什么，为什么它不是“自动加速器”？


CUDA Stream 本质上是任务的顺序队列，用来告诉 GPU 哪些操作按什么顺序执行。

它不是魔法加速器，因为 stream 本身不会让某个 kernel 变快；它的价值在于**组织顺序**和**给并行重叠创造条件**。

如果一个程序里所有任务都排在同一条队列上，很多事情就只能串行执行。只有把任务拆到不同 stream，或者把拷贝、计算、通信放进可重叠的阶段，才有机会把空转时间压下去。


### Q1小验证：stream 解决的是什么问题

先把“队列”这个概念记住，再看重叠就容易多了。

In [ ]:
def stream_timeline(copy_ms, compute_ms, sync_ms):
    """比较串行时间与理想重叠时间；同步仍位于关键路径。"""
    if min(copy_ms, compute_ms, sync_ms) < 0:
        raise ValueError('阶段时间不能为负数')
    serial_ms = copy_ms + compute_ms + sync_ms
    overlapped_ms = max(copy_ms, compute_ms) + sync_ms
    return {
        'serial_ms': serial_ms,
        'overlapped_ms': overlapped_ms,
        'hidden_ms': serial_ms - overlapped_ms,
        'speedup': serial_ms / overlapped_ms if overlapped_ms else 1.0,
    }

cases = [(10, 20, 4), (4, 20, 4), (12, 8, 2)]
for case in cases:
    print(case, '->', stream_timeline(*case))

assert stream_timeline(10, 20, 4)['hidden_ms'] == 10
assert stream_timeline(0, 20, 4)['speedup'] == 1.0


## Q2：为什么异步执行的核心是 overlap，而不是“多开几个任务”？


异步执行的收益不在于“任务更多”，而在于**把原本会等待的时间藏起来**。

常见的重叠包括：
- 数据搬运和计算重叠；
- 前一个 kernel 和后一个 kernel 重叠；
- 通信和计算重叠；
- pipeline 的不同阶段重叠。

如果这些阶段彼此完全依赖，就算开再多 stream 也没意义。真正有价值的异步，是把本来空着的硬件资源填起来，让 GPU 不必在搬运或同步上干等。


### Q2小验证：什么叫 overlap

先分清“并发启动”和“真正重叠”不是一回事。

In [ ]:
def overlap_candidate(copy_ms, compute_ms, event_overhead_ms=0.0, pinned_memory=True):
    """计算可隐藏时间和事件开销；非 pinned host 内存不计入异步拷贝候选。"""
    if min(copy_ms, compute_ms, event_overhead_ms) < 0:
        raise ValueError('时间不能为负数')
    eligible_hidden_ms = min(copy_ms, compute_ms) if pinned_memory else 0.0
    net_saved_ms = eligible_hidden_ms - event_overhead_ms
    return {
        'eligible_hidden_ms': eligible_hidden_ms,
        'event_overhead_ms': event_overhead_ms,
        'net_saved_ms': net_saved_ms,
        'worth_it': net_saved_ms > 0,
    }

for case in [(10, 20, 1), (2, 40, 1), (1, 3, 3)]:
    print(case, '->', overlap_candidate(*case))

assert overlap_candidate(10, 20, 1)['net_saved_ms'] == 9
assert not overlap_candidate(10, 20, 1, pinned_memory=False)['worth_it']


## Q3：为什么调度问题经常比单个算子更影响吞吐？


很多系统慢，不是因为某个算子本身特别差，而是因为调度让大量时间花在等待、同步和小碎操作上。

典型场景包括：
- kernel 太碎，launch 开销被放大；
- 数据搬运没有和计算重叠；
- 任务之间同步太频繁；
- stream 组织不合理，导致本可以并行的阶段被串起来。

所以看吞吐问题时，不能只盯着单个 kernel 的速度，还要看整个执行链是不是被调度拖住了。


### Q3小验证：先判断是不是调度问题

如果单个算子不慢，但整体吞吐差，就先怀疑调度。

In [ ]:
def classify_timeline(kernel_ms, exposed_copy_ms, exposed_sync_ms, idle_ms):
    """按时间线中的暴露时间占比定位主约束，不使用任意压力分数。"""
    parts = {
        'kernel': kernel_ms,
        'copy': exposed_copy_ms,
        'sync': exposed_sync_ms,
        'idle': idle_ms,
    }
    if any(value < 0 for value in parts.values()):
        raise ValueError('时间线字段不能为负数')
    total_ms = sum(parts.values())
    if total_ms == 0:
        raise ValueError('时间线总时间必须大于 0')
    dominant = max(parts, key=parts.get)
    return {
        'total_ms': total_ms,
        'dominant': dominant,
        'fractions': {name: round(value / total_ms, 3) for name, value in parts.items()},
    }

plans = {
    'kernel_bound': (18, 2, 1, 1),
    'sync_bound': (6, 2, 12, 1),
    'copy_bound': (5, 14, 2, 1),
}
for name, timeline in plans.items():
    print(name, '->', classify_timeline(*timeline))

assert classify_timeline(*plans['sync_bound'])['dominant'] == 'sync'


## ⚠️ 常见误区

- stream 不是越多越好，关键是任务是否真的能重叠。
- 异步执行不是取消等待，而是把等待藏到别的工作里。
- 吞吐差不一定是 kernel 慢，也可能是调度方式有问题。
- 先想 overlap，再想优化代码，通常更接近问题本质。

## 相关阅读

**导语：** 把 stream 放进 profiling、缓存访问和调度视角里看，更容易判断吞吐瓶颈是怎么形成的。

- [05. Triton 性能调优与基准测试 (Autotune & Profiling)](../03_Triton_Kernels/05_Triton_Autotune_and_Profiling.ipynb)
- [09. Triton PagedAttention | KV Cache 间接寻址](../03_Triton_Kernels/09_Triton_PagedAttention.ipynb)
- [12. Triton Memory Model and Debug | 内存模型、指针计算与 Debug 避坑指南](../03_Triton_Kernels/12_Triton_Memory_Model_and_Debug.ipynb)
---